### Import Library

In [130]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_val_predict
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error, median_absolute_error, explained_variance_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
import optuna
from optuna.samplers import TPESampler

### Data Loading

In [131]:
df = pd.read_csv('train.csv')
df.shape

(1460, 81)

The dataset contains 1460 entries and 81 columns as follow:

In [132]:
df.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Drop the `Id` column as it is not necessary for model training

In [133]:
df = df.drop(columns='Id')
df.head(5)

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


### Handle Missing Values

In [134]:
df2 = df.copy()

In [ ]:
# Check for missing values
df2_nan = df2.isna().sum()
df2[df2_nan[df2_nan>0].keys()].info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   LotFrontage   1201 non-null   float64
 1   Alley         91 non-null     str    
 2   MasVnrType    588 non-null    str    
 3   MasVnrArea    1452 non-null   float64
 4   BsmtQual      1423 non-null   str    
 5   BsmtCond      1423 non-null   str    
 6   BsmtExposure  1422 non-null   str    
 7   BsmtFinType1  1423 non-null   str    
 8   BsmtFinType2  1422 non-null   str    
 9   Electrical    1459 non-null   str    
 10  FireplaceQu   770 non-null    str    
 11  GarageType    1379 non-null   str    
 12  GarageYrBlt   1379 non-null   float64
 13  GarageFinish  1379 non-null   str    
 14  GarageQual    1379 non-null   str    
 15  GarageCond    1379 non-null   str    
 16  PoolQC        7 non-null      str    
 17  Fence         281 non-null    str    
 18  MiscFeature   54 non-null     str    
d

In [ ]:
# Handle missing values
df2[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']] = df2[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']].fillna(0)
df2['Electrical'] = df2['Electrical'].fillna(df2['Electrical'].mode()[0])

na_features = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
               'BsmtFinType2', 'FireplaceQu', 'PoolQC', 'Fence', 'Alley',
               'MasVnrType', 'GarageType', 'MiscFeature', 'GarageQual',
               'GarageCond', 'GarageFinish']
df2[na_features] = df2[na_features].fillna('NA')

### Encode features

#### Ordinal Encoding

In [137]:
qual_mapping = {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
bsmt_exp_mapping = {'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
bsmt_fin_mapping = {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
func_mapping = {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7}
util_mapping = {'ELO': 0, 'NoSeWa': 1, 'NoSewr': 2, 'AllPub': 3}
binary_mapping = {'N': 0, 'Y': 1}
paved_mapping = {'N': 0, 'P': 1, 'Y': 2}
shape_mapping = {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3}
slope_mapping = {'Sev': 0, 'Mod': 1, 'Gtl': 2}
fence_mapping = {'NA': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
street_alley_mapping = {'NA': 0, 'Grvl': 1, 'Pave': 2}
finish_mapping = {'NA': 0,'Unf': 1, 'RFn': 2, 'Fin': 3}
qual_cond_mapping = {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

mapping_groups = [
    (['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
      'KitchenQual', 'FireplaceQu', 'PoolQC'], qual_mapping),
    (['BsmtExposure'], bsmt_exp_mapping),
    (['BsmtFinType1', 'BsmtFinType2'], bsmt_fin_mapping),
    (['Functional'], func_mapping),
    (['Utilities'], util_mapping),
    (['CentralAir'], binary_mapping),
    (['PavedDrive'], paved_mapping),
    (['LotShape'], shape_mapping),
    (['LandSlope'], slope_mapping),
    (['Fence'], fence_mapping),
    (['Street', 'Alley'], street_alley_mapping),
    (['GarageFinish'], finish_mapping),
    (['GarageQual', 'GarageCond'], qual_cond_mapping)
]

for cols, mapping in mapping_groups:
    for col in cols:
        df2[col] = df2[col].map(mapping)

#### Nominal Encoding

In [138]:
df2['MSSubClass'] = df2['MSSubClass'].astype(str)

nominal_cols = [
    'MSSubClass', 'MSZoning', 'Neighborhood', 'Condition1', 'Condition2',
    'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 
    'Exterior2nd', 'MasVnrType', 'Foundation', 'LandContour', 'LotConfig', 
    'Heating', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition', 'Electrical'
]

df2 = pd.get_dummies(df2, columns=nominal_cols, drop_first=True, dtype=int)


### Model Implimentation

In [ ]:
# TRAIN / TEST SPLIT
X = df2.drop('SalePrice', axis=1)
y = df2['SalePrice']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Initialize XGBoost Regressor model
xgb_model = XGBRegressor(
    n_estimators=100,     
    learning_rate=0.05,   
    max_depth=5,          
    random_state=42,
    n_jobs=-1            
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_valid)

In [ ]:
# Initialize Ridge Regression model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid) 

ridge_model = Ridge(alpha=10.0, random_state=42)

ridge_model.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_model.predict(X_valid_scaled)

### Model Evaluation

We evaluate both models using the following metrics:
 
| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | mean(\|y - ŷ\|) | Average absolute error; easy to interpret in dollar terms |
| **MSE** | mean((y - ŷ)²) | Penalizes large errors more heavily |
| **RMSE** | √MSE | Same unit as target; most common regression metric |
| **MAPE** | mean(\|y - ŷ\| / y) × 100 | Error as a percentage; scale-independent |
| **R²** | 1 - SS_res / SS_tot | Proportion of variance explained; closer to 1 is better |
| **Adjusted R²** | 1 - (1-R²)(n-1)/(n-p-1) | R² penalized for number of features |

In [142]:
def evaluate_model(name, y_true, y_pred, n_features=None):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
 
    # --- Error-based metrics ---
    mae   = mean_absolute_error(y_true, y_pred)
    mse   = mean_squared_error(y_true, y_pred)
    rmse  = np.sqrt(mse)
    r2    = r2_score(y_true, y_pred)
    evs   = explained_variance_score(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
 
    # MAPE
    nonzero_mask = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask])
                          / y_true[nonzero_mask])) * 100
 
    # sMAPE
    smape = np.mean(2 * np.abs(y_pred - y_true)
                    / (np.abs(y_true) + np.abs(y_pred))) * 100
 
    # MSLE
    try:
        msle  = mean_squared_log_error(y_true, y_pred)
        rmsle = np.sqrt(msle)
    except ValueError:
        msle  = np.nan
        rmsle = np.nan
 
    # Adjusted R²
    if n_features is not None:
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    else:
        adj_r2 = np.nan
 
    # Relative metrics
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    rae    = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - np.mean(y_true)))
    rse    = ss_res / ss_tot  # = 1 - R²
 
    print(f"\n{'='*52}")
    print(f"  Model Evaluation: {name}")
    print(f"{'='*52}")
    print(f"\n  [Error-based metrics]")
    print(f"  MAE            : {mae:>15,.2f}  (mean absolute error)")
    print(f"  MedAE          : {medae:>15,.2f}  (median absolute error)")
    print(f"  MSE            : {mse:>15,.2f}  (mean squared error)")
    print(f"  RMSE           : {rmse:>15,.2f}  (root mean squared error)")
    print(f"  MAPE           : {mape:>14.2f}%  (mean absolute % error)")
    print(f"  sMAPE          : {smape:>14.2f}%  (symmetric MAPE)")
    print(f"  RMSLE          : {rmsle:>15.4f}  (log-scale error)")
 
    print(f"\n  [Goodness-of-fit]")
    print(f"  R²             : {r2:>15.4f}  (variance explained)")
    if not np.isnan(adj_r2):
        print(f"  Adjusted R²    : {adj_r2:>15.4f}  (penalizes extra features)")
    print(f"  Explained Var  : {evs:>15.4f}  (like R², ignores bias)")
 
    print(f"\n  [Relative metrics]")
    print(f"  RAE            : {rae:>15.4f}  (relative absolute error)")
    print(f"  RSE            : {rse:>15.4f}  (= 1 - R²)")
    print(f"{'='*52}\n")
 
    return {
        "model": name, "MAE": mae, "MedAE": medae, "MSE": mse,
        "RMSE": rmse, "MAPE": mape, "sMAPE": smape, "RMSLE": rmsle,
        "R2": r2, "Adj_R2": adj_r2, "EVS": evs, "RAE": rae,
    }

In [143]:
n_features = X_train.shape[1]
 
results_xgb   = evaluate_model("XGBoost Regressor", y_valid, y_pred_xgb,   n_features)
results_ridge = evaluate_model("Ridge Regression",  y_valid, y_pred_ridge, n_features)


  Model Evaluation: XGBoost Regressor

  [Error-based metrics]
  MAE            :       16,494.79  (mean absolute error)
  MedAE          :       10,580.16  (median absolute error)
  MSE            :  676,997,888.00  (mean squared error)
  RMSE           :       26,019.18  (root mean squared error)
  MAPE           :           9.99%  (mean absolute % error)
  sMAPE          :           9.29%  (symmetric MAPE)
  RMSLE          :          0.1406  (log-scale error)

  [Goodness-of-fit]
  R²             :          0.9117  (variance explained)
  Adjusted R²    :          0.6664  (penalizes extra features)
  Explained Var  :          0.9117  (like R², ignores bias)

  [Relative metrics]
  RAE            :          0.2665  (relative absolute error)
  RSE            :          0.0883  (= 1 - R²)


  Model Evaluation: Ridge Regression

  [Error-based metrics]
  MAE            :       20,795.04  (mean absolute error)
  MedAE          :       12,990.39  (median absolute error)
  MSE            :

In [144]:
df_results = pd.DataFrame([results_xgb, results_ridge]).set_index("model")
print("\nSummary comparison table:")
print(df_results[["MAE","RMSE","MAPE","sMAPE","R2","Adj_R2"]].round(4).to_string())


Summary comparison table:
                          MAE        RMSE     MAPE    sMAPE      R2  Adj_R2
model                                                                      
XGBoost Regressor  16494.7852  26019.1831   9.9905   9.2913  0.9117  0.6664
Ridge Regression   20795.0355  38291.5109  12.2649  12.2194  0.8088  0.2776


#### K-Fold Cross Validation

**Technical Nature:** K-Fold Cross Validation splits the dataset (`X`, `y`) into $K$ equal parts (folds). The training process occurs over $K$ iterations. In each iteration:
1. One part (the current fold) is held out as the Validation set.
2. The remaining $K-1$ parts are combined to form the Training set.
3. The model is trained on the Training set and evaluated (calculating the RMSE) on the Validation set.

**Purpose:** This mitigates the randomness introduced by a single `train_test_split`. The final result is the arithmetic mean of the $K$ evaluations, providing a highly accurate measure of the model's generalization capabilities and stability across the entire dataset space.

In [145]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_xgb = cross_val_score(xgb_model, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
cv_rmse_xgb = np.sqrt(-cv_scores_xgb)

print("--- K-Fold Cross Validation (XGBoost) ---")
print(f"RMSE each fold: {cv_rmse_xgb}")
print(f"RMSE mean:    {cv_rmse_xgb.mean():,.4f}")
print(f"Standard deviation:      {cv_rmse_xgb.std():,.4f}")

--- K-Fold Cross Validation (XGBoost) ---
RMSE each fold: [26019.18430697 31336.56879749 46380.79326618 28392.93095121
 23635.99018446]
RMSE mean:    31,153.0935
Standard deviation:      8,029.9298


In [146]:
ridge_pipeline = make_pipeline(StandardScaler(), Ridge(alpha=10.0, random_state=42))

cv_scores_ridge = cross_val_score(ridge_pipeline, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
cv_rmse_ridge = np.sqrt(-cv_scores_ridge)

print("\n--- K-Fold Cross Validation (Ridge Regression) ---")
print(f"RMSE each fold: {cv_rmse_ridge}")
print(f"RMSE mean:    {cv_rmse_ridge.mean():,.4f}")
print(f"Standard deviation:      {cv_rmse_ridge.std():,.4f}")


--- K-Fold Cross Validation (Ridge Regression) ---
RMSE each fold: [38291.51093981 33249.38776504 53841.50603103 32835.54677552
 33278.67992951]
RMSE mean:    38,299.3263
Standard deviation:      8,026.4696


### Optimize Model

XGBoost Regressor has shown better results. Therefore, it is selected to go through the model optimization process and will later be used for making final predictions on the test dataset.

In [147]:
def objective(trial):
    selection_threshold = trial.suggest_categorical('feature_threshold', ['median', 'mean', '0.5*mean'])
    feature_selector = SelectFromModel(estimator=xgb_model, threshold=selection_threshold)
    
    param_grid = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1
    }
    
    tuned_xgb = XGBRegressor(**param_grid)
    
    pipeline = Pipeline([
        ('selector', feature_selector),
        ('model', tuned_xgb)
    ])
    
    cv_scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
    return np.sqrt(-cv_scores.mean())

Optuna is an open-source hyperparameter optimization framework. It automates the search for the best model configuration (hyperparameters) to maximize or minimize a specific metric.

In [ ]:
# Suppress default INFO logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Minimize RMSE
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=30)

# Extract hyperparameters
best_threshold = study.best_params.pop('feature_threshold')
best_xgb_params = study.best_params

# Initialize final model
final_selector = SelectFromModel(estimator=xgb_model, threshold=best_threshold)
final_model = XGBRegressor(**best_xgb_params, random_state=42, n_jobs=-1)
final_pipeline = Pipeline([
    ('selector', final_selector),
    ('model', final_model)
])

final_pipeline.fit(X, y);

c:\Users\gtqvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:49:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "feature_threshold" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [149]:
# Evaluate new optimized model
cv_predictions = cross_val_predict(final_pipeline, X, y, cv=kf, n_jobs=-1)

n_final_features = final_pipeline.named_steps['selector'].get_support().sum()

eval_results = evaluate_model("Final Pipeline (5-Fold CV Out-of-Fold)", y, cv_predictions, n_features=n_final_features)


  Model Evaluation: Final Pipeline (5-Fold CV Out-of-Fold)

  [Error-based metrics]
  MAE            :       15,592.59  (mean absolute error)
  MedAE          :        9,398.02  (median absolute error)
  MSE            :  874,354,560.00  (mean squared error)
  RMSE           :       29,569.49  (root mean squared error)
  MAPE           :           9.03%  (mean absolute % error)
  sMAPE          :           8.42%  (symmetric MAPE)
  RMSLE          :          0.1321  (log-scale error)

  [Goodness-of-fit]
  R²             :          0.8614  (variance explained)
  Adjusted R²    :          0.8504  (penalizes extra features)
  Explained Var  :          0.8614  (like R², ignores bias)

  [Relative metrics]
  RAE            :          0.2715  (relative absolute error)
  RSE            :          0.1386  (= 1 - R²)



### Model application

In [150]:
# Load Data
test_data = pd.read_csv('test.csv')
test_ids = test_data['Id']
test_data = test_data.drop(columns=['Id'])

In [151]:
# Check for missing values
test_data.columns[test_data.isna().any()]

Index(['MSZoning', 'LotFrontage', 'Alley', 'Utilities', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath',
       'BsmtHalfBath', 'KitchenQual', 'Functional', 'FireplaceQu',
       'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea',
       'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType'],
      dtype='str')

In [152]:
# Handle missing values
fill_0 = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
          'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
          'GarageCars', 'GarageArea', 'LotFrontage', 'MasVnrArea', 'GarageYrBlt']
test_data[fill_0] = test_data[fill_0].fillna(0)

mode_cols = ['MSZoning', 'Exterior1st', 'Exterior2nd',
             'KitchenQual', 'SaleType', 'Utilities', 'Functional']
for col in mode_cols:
    test_data[col] = test_data[col].fillna(df[col].mode()[0])

test_data[na_features] = test_data[na_features].fillna('NA')

In [153]:
# Ordinal Encoding
for cols, mapping in mapping_groups:
    for col in cols:
        test_data[col] = test_data[col].map(mapping)

# Nominal Encoding
test_data['MSSubClass'] = test_data['MSSubClass'].astype(str)
test_data = pd.get_dummies(test_data, columns=nominal_cols, drop_first=True, dtype=int)

The `reindex` function forces the Test set's columns to perfectly match the Train set by automatically adding missing columns (filled with 0s) and removing extra ones. This ensures complete compatibility with the model and prevents errors when calling predict()

In [154]:
test_data = test_data.reindex(columns=X.columns, fill_value=0)

In [155]:
# Apply model on test.csv
final_preds = final_pipeline.predict(test_data)

In [ ]:
# Export prediction result to csv file
final_submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_preds
})
final_submission.to_csv('submission_final.csv', index=False)